# DOME 改为 Flow Matching 后的逐步探究

这个 notebook 用来帮助你逐步确认本次改动：

1. 检查当前 Python 环境和关键依赖。
2. 读取并确认配置已经切换到 `sample_method='flow'`。
3. 用一个最小 dummy 模型验证 `FlowMatching.training_losses()` 和 `p_sample_loop()`。
4. 标出真实 DOME 模型、VAE、数据和 checkpoint 应该在哪里接入。

建议先在终端激活环境后再启动 Jupyter：

```bash
conda activate madlineCeleste
cd /Users/madlineceleste/Desktop/DOME-main
jupyter notebook flow_matching_model_walkthrough.ipynb
```

In [ ]:
# 基础路径和环境检查
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = Path('/Users/madlineceleste/Desktop/DOME-main')
sys.path.insert(0, str(PROJECT_ROOT))

print('项目路径:', PROJECT_ROOT)
print('Python:', sys.version)
print('解释器:', sys.executable)

# 这些依赖决定能检查到哪一步。
for name in ['torch', 'torchcfm', 'numpy', 'einops', 'mmengine', 'mmcv', 'timm', 'tensorboard', 'xformers']:
    status = 'OK' if importlib.util.find_spec(name) else 'MISSING'
    print(f'{name}: {status}')

## 本次代码改动的核心位置

- `diffusion/flow_matching.py`：新增 flow matching 过程，使用 `torchcfm.conditional_flow_matching.ConditionalFlowMatcher`。
- `diffusion/__init__.py`：新增 `create_flow_matching()` 工厂函数。
- `tools/train_diffusion.py`：训练和验证时根据 `cfg.sample.sample_method` 选择 diffusion 或 flow。
- `tools/eval_metric.py`、`tools/visualize_demo.py`：评估和可视化支持 `sample_method='flow'`。
- `config/train_dome.py`、`config/train_dome_resample.py`：默认切换到 `sample_method='flow'`，并设置 `learn_sigma=False`。

Flow matching 不再预测扩散方差，而是预测速度场 `v_t(x_t)`，所以模型输出通道数应该等于 latent 通道数。

In [ ]:
# 不依赖 mmengine，直接用 runpy 读取 Python 配置文件中的变量。
import runpy

cfg = runpy.run_path(str(PROJECT_ROOT / 'config/train_dome.py'))
sample_cfg = cfg['sample']
world_model_cfg = cfg['model']['world_model']

print('sample_method:', sample_cfg['sample_method'])
print('num_sampling_steps:', sample_cfg['num_sampling_steps'])
print('flow_sigma:', sample_cfg['flow_sigma'])
print('model_time_scale:', sample_cfg['model_time_scale'])
print('learn_sigma:', world_model_cfg['learn_sigma'])
print('in_channels:', world_model_cfg['in_channels'])

assert sample_cfg['sample_method'] == 'flow'
assert world_model_cfg['learn_sigma'] is False
print('配置检查通过：已经切换到 flow matching。')

## 最小化验证 FlowMatching

下面先不用真实 DOME 模型，而是构造一个 dummy 速度场模型。这样可以独立验证新封装的接口是否正确：

- `training_losses()` 应返回 `loss` 和 `mse`。
- loss 的 batch 维度应该是 `(B,)`。
- `p_sample_loop()` 的输出形状应该和输入 latent 形状一致。
- 设置条件帧时，条件帧应该被保留下来。

In [ ]:
import torch
from diffusion import create_flow_matching


class DummyVelocityModel(torch.nn.Module):
    """一个最小速度场模型：只用于接口验证，不代表真实训练效果。"""

    def forward(self, x, t, **kwargs):
        # x: (B, F, C, H, W)
        # t: (B,)，这里 FlowMatching 内部会把 [0, 1] 的 t 放大到 model_time_scale。
        assert x.ndim == 5
        assert t.shape == (x.shape[0],)
        return torch.zeros_like(x)


process = create_flow_matching(
    num_sampling_steps=4,
    sigma=0.0,
    replace_cond_frames=True,
    cond_frames_choices=[[], [0]],
)

model = DummyVelocityModel()
x_start = torch.randn(2, 3, 4, 5, 5)  # B=2, F=3, C=4, H=W=5

losses = process.training_losses(model, x_start, model_kwargs={})
print('loss keys:', sorted(losses.keys()))
print('loss shape:', tuple(losses['loss'].shape))
print('loss mean:', float(losses['loss'].mean()))

assert sorted(losses.keys()) == ['loss', 'mse']
assert tuple(losses['loss'].shape) == (2,)
print('训练 loss 接口检查通过。')

In [ ]:
# 验证采样接口和条件帧替换逻辑。
initial_noise = torch.randn_like(x_start)

sample = process.p_sample_loop(
    model,
    x_start.shape,
    noise=initial_noise,
    device='cpu',
    initial_cond_indices=[0],
    initial_cond_frames=x_start,
)

print('sample shape:', tuple(sample.shape))
print('第 0 帧是否保持为条件帧:', torch.allclose(sample[:, 0], x_start[:, 0]))

assert sample.shape == x_start.shape
assert torch.allclose(sample[:, 0], x_start[:, 0])
print('采样接口和条件帧逻辑检查通过。')

## 真实 DOME 模型的接入点

真实训练时，`tools/train_diffusion.py` 的关键流程是：

1. VAE 把 occupancy 编码成 latent：`vae.forward_encoder()` 和 `vae.sample_z()`。
2. latent 被整理为 `(B, F, C, H, W)`。
3. `FlowMatching.training_losses(my_model, x, model_kwargs=...)` 采样 `(t, x_t, u_t)`。
4. DOME 预测速度场 `v_t`。
5. loss 为 `MSE(v_t, u_t)`。

真实评估时，`p_sample_loop()` 从高斯噪声开始做 ODE 积分，并在每一步把历史条件帧强制替换回真实 latent。

In [ ]:
# 可选：如果当前环境已经装好 mmengine/mmcv/timm 等依赖，可以尝试构建真实 DOME 模型。
# 当前 madlineCeleste 环境如果缺依赖，本单元会给出缺失项，而不会中断整个 notebook。
import importlib.util

required_for_real_model = ['mmengine', 'mmcv', 'timm', 'einops']
missing = [name for name in required_for_real_model if importlib.util.find_spec(name) is None]

if missing:
    print('暂时不能构建真实 DOME 模型，缺少依赖:', missing)
    print('先安装项目环境依赖后，再重新运行这个单元。')
else:
    from mmengine import Config
    from mmengine.registry import MODELS
    import model  # 注册 DOME / VAE 模型

    mm_cfg = Config.fromfile(str(PROJECT_ROOT / 'config/train_dome.py'))
    dome = MODELS.build(mm_cfg.model.world_model)
    print(dome)
    print('真实 DOME 模型构建成功。')

## 真实数据单 batch 验证模板

下面这个单元是模板，不会默认运行完整训练。等数据、VAE checkpoint、DOME 依赖都准备好以后，可以按需取消注释。目标是只取一个 batch，验证完整链路能跑到 flow matching loss。

In [ ]:
# 真实单 batch 验证模板：准备好依赖、数据和 ckpt 后再取消注释运行。
#
# from mmengine import Config
# from mmengine.registry import MODELS
# from einops import rearrange
# import model
# from dataset import get_dataloader
# from diffusion import create_flow_matching
#
# cfg = Config.fromfile(str(PROJECT_ROOT / 'config/train_dome.py'))
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
#
# dome = MODELS.build(cfg.model.world_model).to(device).train()
# vae = MODELS.build(cfg.model.vae).to(device).eval()
# vae.requires_grad_(False)
#
# process = create_flow_matching(
#     num_sampling_steps=cfg.sample.num_sampling_steps,
#     sigma=cfg.sample.flow_sigma,
#     replace_cond_frames=cfg.replace_cond_frames,
#     cond_frames_choices=cfg.cond_frames_choices,
#     model_time_scale=cfg.sample.model_time_scale,
# )
#
# train_loader, _ = get_dataloader(
#     cfg.train_dataset_config,
#     cfg.val_dataset_config,
#     cfg.train_wrapper_config,
#     cfg.val_wrapper_config,
#     cfg.train_loader,
#     cfg.val_loader,
#     dist=False,
# )
#
# input_occs, target_occs, metas = next(iter(train_loader))
# input_occs = input_occs.to(device)
# bs = input_occs.shape[0]
#
# with torch.no_grad():
#     latents, shapes = vae.forward_encoder(input_occs)
#     latents, _, _ = vae.sample_z(latents)
#     latents = latents * cfg.model.vae.scaling_factor
#     if latents.dim() == 4:
#         latents = rearrange(latents, '(b f) c h w -> b f c h w', b=bs).contiguous()
#     elif latents.dim() == 5:
#         latents = rearrange(latents, 'b c f h w -> b f c h w', b=bs).contiguous()
#     else:
#         raise NotImplementedError(latents.shape)
#
# loss_dict = process.training_losses(dome, latents, model_kwargs={'metas': metas})
# print({k: float(v.mean()) for k, v in loss_dict.items()})

## 当前验证结论

如果前面的 dummy 单元都通过，说明本次新增的 flow matching 包装层可以正常完成：

- 从 `torchcfm` 采样训练用的 `(t, x_t, u_t)`。
- 调用 DOME 风格的 `model(x_t, t, **model_kwargs)`。
- 计算速度场 MSE loss。
- 用 ODE 采样生成 latent。
- 在采样过程中保留历史条件帧。

真正开始训练前，还需要确保当前环境补齐 DOME 原项目依赖，例如 `mmengine`、`mmcv`、`timm`、`einops`、`tensorboard`、`xformers`，并确认它们和当前 `torch` 版本兼容。